# RT-DETRv4 Colab Notebook (Inpainting Augmentation Included)

이 노트북은 **Colab**에서 바로 실행할 수 있게 구성되어 있다.

포함된 작업:
1. 환경 설치
2. RT-DETRv4 repo clone
3. 데이터셋 읽기 (`train_images`, `train_annotations`)
4. **annotation 하나씩 제거하는 inpainting 증강**
5. COCO 포맷 변환
6. RT-DETRv4 config 패치
7. Colab에서 바로 학습 실행

전제:
- 데이터셋 루트 안에 `train_images`, `train_annotations`가 있음
- annotation 구조는 이전에 맞춘 형태 기준
- Colab GPU 런타임 사용 권장


## 1) 환경 설치
처음 한 번 실행하면 된다.


In [1]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q pyyaml opencv-python-headless tqdm faster-coco-eval

Fri Mar 27 13:48:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.74                 Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1660 ...  WDDM  |   00000000:06:00.0  On |                  N/A |
| 50%   41C    P0             42W /  125W |    1266MiB /   6144MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ERROR: Could not install packages due to an EnvironmentError: [WinError 5] 액세스가 거부되었습니다: 'c:\\programdata\\anaconda3\\lib\\site-packages\\pip\\_internal\\build_env.py'
Consider using the `--user` option or check the permissions.



## 2) Google Drive 마운트 (선택)
데이터셋이 Drive에 있으면 이 셀을 사용한다.


In [40]:
import zipfile
import os

dataset_path = 'C:/Users/USER/Downloads/ai09-level1-project.zip'
output_dir = 'C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path) as zip_file:
    zip_file.extractall(path=output_dir)

data_root = os.path.join(output_dir, 'sprint_ad_project1_data')
train_image_dir = os.path.join(data_root, "train_images")
train_annotation_dir = os.path.join(data_root, "train_annotations")
test_image_dir = os.path.join(data_root, "test_images")

print("Train image dir:", train_image_dir)
print("Train annotation dir:", train_annotation_dir)
print("Test image dir:", test_image_dir)

Train image dir: C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/sprint_ad_project1_data\train_images
Train annotation dir: C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/sprint_ad_project1_data\train_annotations
Test image dir: C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/sprint_ad_project1_data\test_images


In [41]:
from pathlib import Path

DATA_ROOT = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/sprint_ai_project1_data")

IMAGE_DIR_NAME = "train_images"
ANNOTATION_DIR_NAME = "train_annotations"

TRAIN_IMAGE_DIR = DATA_ROOT / IMAGE_DIR_NAME
TRAIN_ANNOTATION_DIR = DATA_ROOT / ANNOTATION_DIR_NAME

RTDETR_REPO = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4")
BASE_CONFIG = RTDETR_REPO / "configs" / "rtv4" / "rtv4_hgnetv2_x_coco.yml"
PATCHED_CONFIG_OUT = RTDETR_REPO / "configs" / "rtv4" / "custom_rtdetrv4_config.yml"

DINOV3_REPO_PATH = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/dinov3")
DINOV3_WEIGHTS_PATH = DINOV3_REPO_PATH / "dinov3" / "weights" / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"

OUTPUT_ROOT = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/rtdetrv4_work")

COPY_ORIGINALS = True
OVERWRITE = True
SEED = 42

PER_ANN_AUG_RATIO = 0.5
MAX_AUG_PER_IMAGE = 1
MAJORITY_ONLY = True
MAJORITY_TOP_PERCENT = 0.30
EXPAND_RATIO = 0.08
INPAINT_RADIUS = 7
INPAINT_METHOD = "telea"   # "telea" or "ns"
AUG_SUFFIX = "_rm"

VAL_RATIO = 0.2
RESIZE = 640   # Colab OOM 방지를 위해 480부터 시작 추천

NUM_CLASSES = 56
TOTAL_BATCH_SIZE = 2
NUM_WORKERS = 2
USE_AMP = True
RESUME_CHECKPOINT = None

print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR, TRAIN_IMAGE_DIR.exists())
print("TRAIN_ANNOTATION_DIR:", TRAIN_ANNOTATION_DIR, TRAIN_ANNOTATION_DIR.exists())
print("RTDETR_REPO:", RTDETR_REPO, RTDETR_REPO.exists())
print("BASE_CONFIG:", BASE_CONFIG, BASE_CONFIG.exists())

TRAIN_IMAGE_DIR: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\ai09-project01\sprint_ai_project1_data\train_images True
TRAIN_ANNOTATION_DIR: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\ai09-project01\sprint_ai_project1_data\train_annotations True
RTDETR_REPO: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\RT-DETRv4 True
BASE_CONFIG: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\RT-DETRv4\configs\rtv4\rtv4_hgnetv2_x_coco.yml True


## 5) 공통 import 및 유틸 함수


In [42]:
import os
import sys
import json
import yaml
import cv2
import shutil
import random
import subprocess
from copy import deepcopy
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from tqdm import tqdm

random.seed(SEED)
np.random.seed(SEED)

OUT_RAW_ROOT = OUTPUT_ROOT / "augmented_raw"
OUT_IMAGE_DIR = OUT_RAW_ROOT / "train_images"
OUT_ANNOTATION_DIR = OUT_RAW_ROOT / "train_annotations"

COCO_ROOT = OUTPUT_ROOT / "coco_dataset"
COCO_TRAIN_IMAGE_DIR = COCO_ROOT / "images" / "train"
COCO_VAL_IMAGE_DIR = COCO_ROOT / "images" / "val"
COCO_ANN_DIR = COCO_ROOT / "annotations"
COCO_TRAIN_JSON = COCO_ANN_DIR / "instances_train.json"
COCO_VAL_JSON = COCO_ANN_DIR / "instances_val.json"

IMG_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".webp"]

def ensure_clean_dir(path: Path, overwrite=False):
    if path.exists() and overwrite:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def find_image_files(image_dir: Path):
    files = []
    for ext in IMG_EXTS:
        files.extend(image_dir.glob(f"*{ext}"))
    return sorted(files)

def find_json_files(annotation_dir: Path):
    return sorted(annotation_dir.rglob("*.json"))

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=True, indent=2)

def get_image_filename_from_annotation(data, fallback_stem=None):
    images = data.get("images", [])
    if isinstance(images, list) and len(images) > 0:
        file_name = images[0].get("file_name", None)
        if file_name:
            return os.path.basename(file_name)
    if fallback_stem is not None:
        for ext in IMG_EXTS:
            return f"{fallback_stem}{ext}"
    return None

def match_image_path(image_dir: Path, file_name=None, stem=None):
    if file_name is not None:
        p = image_dir / os.path.basename(file_name)
        if p.exists():
            return p

    if stem is not None:
        for ext in IMG_EXTS:
            p = image_dir / f"{stem}{ext}"
            if p.exists():
                return p

    if file_name is not None:
        wanted_stem = Path(file_name).stem
        for ext in IMG_EXTS:
            p = image_dir / f"{wanted_stem}{ext}"
            if p.exists():
                return p

    return None

def build_mask_from_single_annotation(annotation, image_shape, expand_ratio=0.08):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    x, y, bw, bh = annotation["bbox"]  # COCO xywh
    x1 = max(0, int(x - bw * expand_ratio))
    y1 = max(0, int(y - bh * expand_ratio))
    x2 = min(w - 1, int(x + bw + bw * expand_ratio))
    y2 = min(h - 1, int(y + bh + bh * expand_ratio))

    mask[y1:y2, x1:x2] = 255
    return mask

def inpaint_method_from_name(name: str):
    name = name.lower().strip()
    if name == "telea":
        return cv2.INPAINT_TELEA
    if name == "ns":
        return cv2.INPAINT_NS
    raise ValueError("INPAINT_METHOD must be 'telea' or 'ns'")

## 6) 원본 annotation 읽기
JSON 내부의 `images[0]['file_name']` 기준으로 이미지 단위로 묶는다.


In [43]:
def extract_dataset_records(image_dir: Path, annotation_dir: Path):
    grouped = defaultdict(list)
    json_paths = find_json_files(annotation_dir)

    for json_path in tqdm(json_paths, desc="Grouping json files by true image file"):
        data = load_json(json_path)
        file_name = get_image_filename_from_annotation(data, fallback_stem=json_path.stem)
        if file_name is None:
            continue

        image_key = Path(file_name).stem
        grouped[image_key].append((json_path, data, file_name))

    records = []
    for image_key, items in tqdm(grouped.items(), desc="Loading grouped dataset records"):
        representative_json_path, representative_data, representative_file_name = items[0]
        img_path = match_image_path(image_dir, file_name=representative_file_name, stem=image_key)

        merged_annotations = []
        merged_annotation_dicts = []

        for json_path, data, file_name in items:
            anns = data.get("annotations", [])
            for ann in anns:
                bbox = ann.get("bbox", None)
                cat_id = ann.get("category_id", None)

                if isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                    merged_annotations.append({
                        "bbox": bbox,
                        "category_id": cat_id,
                        "annotation_id": ann.get("id", None),
                        "source_json": str(json_path)
                    })

                    ann_copy = deepcopy(ann)
                    ann_copy["_source_json"] = str(json_path)
                    ann_copy["_source_relpath"] = str(Path(json_path).relative_to(annotation_dir))
                    merged_annotation_dicts.append(ann_copy)

        merged_data = deepcopy(representative_data)
        merged_data["annotations"] = merged_annotation_dicts
        if "images" not in merged_data or not merged_data["images"]:
            merged_data["images"] = [{"file_name": representative_file_name}]
        else:
            merged_data["images"][0]["file_name"] = representative_file_name

        records.append({
            "image_key": image_key,
            "json_paths": [str(x[0]) for x in items],
            "image_path": img_path,
            "image_file_name": representative_file_name,
            "data": merged_data,
            "annotations": merged_annotations
        })

    return records

records = extract_dataset_records(TRAIN_IMAGE_DIR, TRAIN_ANNOTATION_DIR)
print("총 record 수:", len(records))

if len(records) > 0:
    ann_counts = [len(r["annotations"]) for r in records]
    print("annotation 개수 통계")
    print("min:", min(ann_counts))
    print("max:", max(ann_counts))
    print("mean:", sum(ann_counts) / len(ann_counts))

    unmatched = [r for r in records if r["image_path"] is None]
    print("이미지 매칭 실패 수:", len(unmatched))
    if unmatched:
        print("예시 unmatched image key:", unmatched[0]["image_key"])
        print("예시 unmatched image file:", unmatched[0]["image_file_name"])
else:
    print("records가 비어 있음. 경로를 먼저 확인해야 함.")

Loading grouped dataset records: 100%|██████████| 232/232 [00:00<00:00, 3177.36it/s]

총 record 수: 232
annotation 개수 통계
min: 2
max: 4
mean: 3.288793103448276
이미지 매칭 실패 수: 0


In [44]:
def build_master_category_mapping(annotation_dir: Path):
    category_bank = {}

    for json_path in tqdm(find_json_files(annotation_dir), desc="Collecting master categories"):
        data = load_json(json_path)
        for cat in data.get("categories", []):
            old_id = cat.get("id", None)
            if old_id is None:
                continue
            if old_id not in category_bank:
                category_bank[old_id] = deepcopy(cat)

    old_ids_sorted = sorted(category_bank.keys())
    old_to_new = {old_id: new_id for new_id, old_id in enumerate(old_ids_sorted)}

    master_categories = []
    for old_id in old_ids_sorted:
        cat = category_bank[old_id]
        master_categories.append({
            "id": old_to_new[old_id],
            "name": cat.get("name", str(old_id))
        })

    return old_to_new, master_categories, category_bank

OLD_CAT_ID_TO_NEW_ID, MASTER_CATEGORIES, MASTER_CATEGORY_BANK = build_master_category_mapping(TRAIN_ANNOTATION_DIR)

print("원본 전체 categories 수:", len(MASTER_CATEGORIES))
print("예시 master categories:", MASTER_CATEGORIES[:10])

assert len(MASTER_CATEGORIES) == 56, f"56개가 아니라 {len(MASTER_CATEGORIES)}개입니다. 원본 categories를 다시 확인해야 합니다."

NUM_CLASSES = 56

원본 전체 categories 수: 56
예시 master categories: [{'id': 0, 'name': '보령부스파정 5mg'}, {'id': 1, 'name': '뮤테란캡슐 100mg'}, {'id': 2, 'name': '일양하이트린정 2mg'}, {'id': 3, 'name': '기넥신에프정(은행엽엑스)(수출용)'}, {'id': 4, 'name': '무코스타정(레바미피드)(비매품)'}, {'id': 5, 'name': '알드린정'}, {'id': 6, 'name': '뉴로메드정(옥시라세탐)'}, {'id': 7, 'name': '에어탈정(아세클로페낙)'}, {'id': 8, 'name': '리렉스펜정 300mg/PTP'}, {'id': 9, 'name': '아빌리파이정 10mg'}]


## 7) 클래스 분포 계산 및 다수 클래스 선정


In [45]:
def compute_class_counter(records):
    class_counter = Counter()
    for r in records:
        for ann in r["annotations"]:
            cid = ann.get("category_id", None)
            if cid is not None:
                class_counter[cid] += 1
    return class_counter

class_counter = compute_class_counter(records)
print("클래스별 객체 수 상위 20개:")
for cls_id, cnt in class_counter.most_common(20):
    print(cls_id, cnt)

sorted_classes = [cls for cls, _ in class_counter.most_common()]
top_k = max(1, int(len(sorted_classes) * MAJORITY_TOP_PERCENT))
majority_classes = set(sorted_classes[:top_k])

print("\n다수 클래스로 사용할 클래스들:")
print(sorted(list(majority_classes))[:30], "...")
print("총 개수:", len(majority_classes))

클래스별 객체 수 상위 20개:
3351 153
3483 45
35206 37
16262 23
21325 22
16232 21
3832 20
20238 20
36637 19
16548 18
29667 18
38162 18
19232 17
22074 16
20014 16
33880 16
41768 16
1900 15
13900 15
18147 15

다수 클래스로 사용할 클래스들:
[3351, 3483, 3832, 16232, 16262, 16548, 19232, 20014, 20238, 21325, 22074, 29667, 33880, 35206, 36637, 38162] ...
총 개수: 16


## 8) annotation 하나씩 제거하는 inpainting 증강 함수


In [46]:
def get_removable_annotation_indices(annotations, majority_classes=None, majority_only=True):
    removable = []
    for i, ann in enumerate(annotations):
        cid = ann.get("category_id", None)
        if majority_only:
            if cid in majority_classes:
                removable.append(i)
        else:
            removable.append(i)
    return removable

def save_augmented_record_per_annotation(
    record,
    out_img_dir,
    out_ann_dir,
    majority_classes,
    majority_only=True,
    expand_ratio=0.08,
    inpaint_radius=7,
    inpaint_method=cv2.INPAINT_TELEA,
    aug_suffix="_rm",
    max_aug_per_image=None,
):
    if record["image_path"] is None or not Path(record["image_path"]).exists():
        return 0, ["image_not_found"]

    image = cv2.imread(str(record["image_path"]))
    if image is None:
        return 0, ["image_read_fail"]

    annotations = deepcopy(record["data"].get("annotations", []))
    if len(annotations) <= 1:
        return 0, ["not_enough_annotations"]

    removable_indices = get_removable_annotation_indices(
        annotations=annotations,
        majority_classes=majority_classes,
        majority_only=majority_only
    )
    if len(removable_indices) == 0:
        return 0, ["no_removable_annotations"]

    if max_aug_per_image is not None:
        removable_indices = removable_indices[:max_aug_per_image]

    success_count = 0
    fail_reasons = []

    orig_name = Path(record["image_file_name"]).name
    orig_stem = Path(orig_name).stem
    orig_suffix = Path(orig_name).suffix

    for ann_idx in removable_indices:
        ann_to_remove = annotations[ann_idx]
        kept_annotations = [deepcopy(ann) for i, ann in enumerate(annotations) if i != ann_idx]

        if len(kept_annotations) == 0:
            fail_reasons.append("all_annotations_removed")
            continue

        mask = build_mask_from_single_annotation(
            annotation=ann_to_remove,
            image_shape=image.shape,
            expand_ratio=expand_ratio
        )
        aug_image = cv2.inpaint(image, mask, inpaint_radius, inpaint_method)

        remove_cls = ann_to_remove.get("category_id", "unk")
        new_img_name = f"{orig_stem}{aug_suffix}_ann{ann_idx:03d}_cls{remove_cls}{orig_suffix}"
        out_img_path = Path(out_img_dir) / new_img_name
        out_img_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_img_path), aug_image)

        image_json_root = Path(out_ann_dir) / f"{Path(new_img_name).stem}_json"
        image_json_root.mkdir(parents=True, exist_ok=True)

        grouped_by_source = {}
        for ann in kept_annotations:
            src_rel = ann.get("_source_relpath", None)
            src_json = ann.get("_source_json", None)

            if src_rel is None and src_json is not None:
                src_rel = Path(src_json).name
            if src_rel is None:
                src_rel = "unknown/ann.json"

            grouped_by_source.setdefault(src_rel, []).append(ann)

        for src_rel, anns_in_one_json in grouped_by_source.items():
            src_rel = Path(src_rel)
            rel_parts = src_rel.parts

            if len(rel_parts) >= 2:
                relative_inside_image_json = Path(*rel_parts[1:])
            else:
                relative_inside_image_json = Path(src_rel.name)

            out_json_path = image_json_root / relative_inside_image_json
            out_json_path.parent.mkdir(parents=True, exist_ok=True)

            json_data = {
                "images": deepcopy(record["data"].get("images", [])),
                "annotations": [],
                "categories": deepcopy(record["data"].get("categories", []))
            }

            if not json_data["images"]:
                json_data["images"] = [{"file_name": new_img_name}]
            else:
                json_data["images"][0]["file_name"] = new_img_name

            cleaned_anns = []
            for ann in anns_in_one_json:
                ann_copy = deepcopy(ann)
                ann_copy.pop("_source_json", None)
                ann_copy.pop("_source_relpath", None)
                cleaned_anns.append(ann_copy)

            json_data["annotations"] = cleaned_anns
            save_json(out_json_path, json_data)

        success_count += 1

    return success_count, fail_reasons

## 9) 원본 복사 + 증강 생성 실행


In [47]:
if OVERWRITE:
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

ensure_clean_dir(OUT_IMAGE_DIR, overwrite=False)
ensure_clean_dir(OUT_ANNOTATION_DIR, overwrite=False)

if COPY_ORIGINALS:
    for r in tqdm(records, desc="Copying originals"):
        if r["image_path"] is not None and Path(r["image_path"]).exists():
            shutil.copy2(r["image_path"], OUT_IMAGE_DIR / Path(r["image_path"]).name)

        ann_subdir = OUT_ANNOTATION_DIR / f"{Path(r['image_file_name']).stem}_json"
        ann_subdir.mkdir(parents=True, exist_ok=True)

        for jp in r["json_paths"]:
            jp = Path(jp)
            shutil.copy2(jp, ann_subdir / jp.name)

num_to_augment = int(len(records) * PER_ANN_AUG_RATIO)
selected_records = random.sample(records, num_to_augment)

total_success = 0
fail_stats = Counter()

for r in tqdm(selected_records, desc="Generating per-annotation remove-mask samples"):
    success_count, reasons = save_augmented_record_per_annotation(
        record=r,
        out_img_dir=OUT_IMAGE_DIR,
        out_ann_dir=OUT_ANNOTATION_DIR,
        majority_classes=majority_classes,
        majority_only=MAJORITY_ONLY,
        expand_ratio=EXPAND_RATIO,
        inpaint_radius=INPAINT_RADIUS,
        inpaint_method=inpaint_method_from_name(INPAINT_METHOD),
        aug_suffix=AUG_SUFFIX,
        max_aug_per_image=MAX_AUG_PER_IMAGE
    )
    total_success += success_count
    for reason in reasons:
        fail_stats[reason] += 1

print("원본 record 수:", len(records))
print("증강 대상 record 수:", len(selected_records))
print("생성된 증강 이미지 수:", total_success)
print("실패 통계:", dict(fail_stats))
print("증강 결과 폴더:", OUT_RAW_ROOT)

Generating per-annotation remove-mask samples: 100%|██████████| 116/116 [00:00<00:00, 6443.29it/s]

원본 record 수: 232
증강 대상 record 수: 116
생성된 증강 이미지 수: 0
실패 통계: {'image_read_fail': 116}
증강 결과 폴더: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\rtdetrv4_work\augmented_raw


## 10) COCO 변환
RT-DETRv4 학습용으로 바로 사용할 수 있게 COCO 포맷으로 변환한다.


In [33]:
def load_grouped_record_from_augmented_pair(image_path: Path, ann_root_dir: Path):
    ann_jsons = sorted(ann_root_dir.rglob("*.json"))
    if len(ann_jsons) == 0:
        return None

    merged = {
        "images": [{"file_name": image_path.name}],
        "annotations": [],
        "categories": []
    }
    category_map = {}

    for jp in ann_jsons:
        data = load_json(jp)
        for ann in data.get("annotations", []):
            merged["annotations"].append(deepcopy(ann))
        for cat in data.get("categories", []):
            cid = cat.get("id")
            if cid not in category_map:
                category_map[cid] = deepcopy(cat)

    merged["categories"] = list(category_map.values())
    return merged

def collect_augmented_pairs(out_image_dir: Path, out_ann_dir: Path):
    image_files = find_image_files(out_image_dir)
    pairs = []

    for img_path in image_files:
        ann_root = out_ann_dir / f"{img_path.stem}_json"
        if not ann_root.exists():
            continue
        merged_data = load_grouped_record_from_augmented_pair(img_path, ann_root)
        if merged_data is None:
            continue
        pairs.append({
            "image_path": img_path,
            "ann_root": ann_root,
            "data": merged_data
        })
    return pairs

aug_pairs = collect_augmented_pairs(OUT_IMAGE_DIR, OUT_ANNOTATION_DIR)
print("COCO 변환 대상 pair 수:", len(aug_pairs))
print("예시 이미지:", aug_pairs[0]["image_path"].name if aug_pairs else None)

COCO 변환 대상 pair 수: 232
예시 이미지: K-001900-016548-019607-029451_0_2_0_2_70_000_200.png


In [34]:
def split_pairs_train_val(pairs, val_ratio=0.2, seed=42):
    pairs = pairs.copy()
    rng = random.Random(seed)
    rng.shuffle(pairs)

    n_val = max(1, int(len(pairs) * val_ratio))
    val_pairs = pairs[:n_val]
    train_pairs = pairs[n_val:]
    return train_pairs, val_pairs

train_pairs, val_pairs = split_pairs_train_val(aug_pairs, val_ratio=VAL_RATIO, seed=SEED)
print("train pairs:", len(train_pairs))
print("val pairs:", len(val_pairs))

train pairs: 186
val pairs: 46


In [35]:
def maybe_resize_image_and_boxes(image, anns, resize=None):
    if resize is None:
        return image, anns

    h, w = image.shape[:2]
    target = int(resize)

    scale = min(target / max(h, w), target / max(h, w))
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    new_anns = []
    for ann in anns:
        ann_copy = deepcopy(ann)
        x, y, bw, bh = ann_copy["bbox"]
        ann_copy["bbox"] = [x * scale, y * scale, bw * scale, bh * scale]
        ann_copy["area"] = float(ann_copy["bbox"][2] * ann_copy["bbox"][3])
        new_anns.append(ann_copy)

    return resized, new_anns


def export_pairs_to_coco(
    pairs,
    image_out_dir: Path,
    coco_json_path: Path,
    category_id_map: dict,
    master_categories: list,
    resize=None
):
    image_out_dir.mkdir(parents=True, exist_ok=True)
    coco_json_path.parent.mkdir(parents=True, exist_ok=True)

    images = []
    annotations = []
    image_id = 1
    ann_id = 1

    skipped_unknown_category = 0
    skipped_invalid_bbox = 0

    for pair in tqdm(pairs, desc=f"Exporting {coco_json_path.name}"):
        img_path = pair["image_path"]
        data = pair["data"]

        image = cv2.imread(str(img_path))
        if image is None:
            continue

        anns = deepcopy(data.get("annotations", []))
        image_resized, anns_resized = maybe_resize_image_and_boxes(image, anns, resize=resize)

        out_img_path = image_out_dir / img_path.name
        cv2.imwrite(str(out_img_path), image_resized)

        h, w = image_resized.shape[:2]
        images.append({
            "id": image_id,
            "file_name": img_path.name,
            "width": w,
            "height": h,
        })

        for ann in anns_resized:
            old_cid = ann.get("category_id", None)
            bbox = ann.get("bbox", None)

            if old_cid is None or bbox is None:
                skipped_invalid_bbox += 1
                continue

            if old_cid not in category_id_map:
                skipped_unknown_category += 1
                continue

            if not (isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0):
                skipped_invalid_bbox += 1
                continue

            new_cid = category_id_map[old_cid]

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": new_cid,
                "bbox": [float(v) for v in bbox],
                "area": float(bbox[2] * bbox[3]),
                "iscrowd": int(ann.get("iscrowd", 0)),
            })
            ann_id += 1

        image_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": deepcopy(master_categories)
    }
    save_json(coco_json_path, coco)

    print(f"[{coco_json_path.name}] images:", len(images))
    print(f"[{coco_json_path.name}] annotations:", len(annotations))
    print(f"[{coco_json_path.name}] categories:", len(master_categories))
    print(f"[{coco_json_path.name}] skipped_unknown_category:", skipped_unknown_category)
    print(f"[{coco_json_path.name}] skipped_invalid_bbox:", skipped_invalid_bbox)

    return coco


ensure_clean_dir(COCO_TRAIN_IMAGE_DIR, overwrite=True)
ensure_clean_dir(COCO_VAL_IMAGE_DIR, overwrite=True)
ensure_clean_dir(COCO_ANN_DIR, overwrite=False)

coco_train = export_pairs_to_coco(
    train_pairs,
    COCO_TRAIN_IMAGE_DIR,
    COCO_TRAIN_JSON,
    category_id_map=OLD_CAT_ID_TO_NEW_ID,
    master_categories=MASTER_CATEGORIES,
    resize=RESIZE
)

coco_val = export_pairs_to_coco(
    val_pairs,
    COCO_VAL_IMAGE_DIR,
    COCO_VAL_JSON,
    category_id_map=OLD_CAT_ID_TO_NEW_ID,
    master_categories=MASTER_CATEGORIES,
    resize=RESIZE
)

print("COCO train images:", len(coco_train["images"]))
print("COCO train anns:", len(coco_train["annotations"]))
print("COCO train categories:", len(coco_train["categories"]))
print("COCO val images:", len(coco_val["images"]))
print("COCO val anns:", len(coco_val["annotations"]))
print("COCO val categories:", len(coco_val["categories"]))

Exporting instances_train.json: 100%|██████████| 186/186 [00:00<00:00, 92907.06it/s]


[instances_train.json] images: 0
[instances_train.json] annotations: 0
[instances_train.json] categories: 56
[instances_train.json] skipped_unknown_category: 0
[instances_train.json] skipped_invalid_bbox: 0


Exporting instances_val.json: 100%|██████████| 46/46 [00:00<00:00, 46036.26it/s]

[instances_val.json] images: 0
[instances_val.json] annotations: 0
[instances_val.json] categories: 56
[instances_val.json] skipped_unknown_category: 0
[instances_val.json] skipped_invalid_bbox: 0
COCO train images: 0
COCO train anns: 0
COCO train categories: 56
COCO val images: 0
COCO val anns: 0
COCO val categories: 56


## 11) RT-DETRv4 config 패치
Colab GPU 메모리를 고려해서 기본값은 **small + batch 1 + workers 2 + resize 480** 기준으로 맞춘다.


In [36]:
def patch_rtdetr_config(
    base_config_path: Path,
    output_config_path: Path,
    train_img_dir: Path,
    train_ann_file: Path,
    val_img_dir: Path,
    val_ann_file: Path,
    num_classes: int,
    dinov3_repo_path=None,
    dinov3_weights_path=None,
):
    with open(base_config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["num_classes"] = int(num_classes)
    cfg["remap_mscoco_category"] = False
    cfg["sync_bn"] = False

    cfg.setdefault("train_dataloader", {})
    cfg["train_dataloader"].setdefault("dataset", {})
    cfg["train_dataloader"]["dataset"]["img_folder"] = str(train_img_dir)
    cfg["train_dataloader"]["dataset"]["ann_file"] = str(train_ann_file)
    cfg["train_dataloader"]["total_batch_size"] = TOTAL_BATCH_SIZE
    cfg["train_dataloader"]["num_workers"] = NUM_WORKERS

    cfg.setdefault("val_dataloader", {})
    cfg["val_dataloader"].setdefault("dataset", {})
    cfg["val_dataloader"]["dataset"]["img_folder"] = str(val_img_dir)
    cfg["val_dataloader"]["dataset"]["ann_file"] = str(val_ann_file)
    cfg["val_dataloader"]["total_batch_size"] = 1
    cfg["val_dataloader"]["num_workers"] = NUM_WORKERS

    def patch_resize_in_dataset(dataset_cfg, target_size=480):
        transforms_cfg = dataset_cfg.get("transforms", None)
        if transforms_cfg is None:
            return

        if isinstance(transforms_cfg, dict) and "ops" in transforms_cfg:
            for op in transforms_cfg["ops"]:
                if isinstance(op, dict) and op.get("type") == "Resize":
                    op["size"] = [target_size, target_size]
        elif isinstance(transforms_cfg, list):
            for op in transforms_cfg:
                if isinstance(op, dict) and op.get("type") == "Resize":
                    op["size"] = [target_size, target_size]

    patch_resize_in_dataset(cfg["train_dataloader"]["dataset"], target_size=RESIZE)
    patch_resize_in_dataset(cfg["val_dataloader"]["dataset"], target_size=RESIZE)

    if dinov3_repo_path is not None:
        cfg["dinov3_repo_path"] = str(dinov3_repo_path)
    if dinov3_weights_path is not None:
        cfg["dinov3_pretrain_weights"] = str(dinov3_weights_path)

    if "teacher_model" in cfg and isinstance(cfg["teacher_model"], dict):
        if dinov3_repo_path is not None:
            cfg["teacher_model"]["dinov3_repo_path"] = str(dinov3_repo_path)
        if dinov3_weights_path is not None:
            cfg["teacher_model"]["dinov3_weights_path"] = str(dinov3_weights_path)

    with open(output_config_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

patch_rtdetr_config(
    base_config_path=BASE_CONFIG,
    output_config_path=PATCHED_CONFIG_OUT,
    train_img_dir=COCO_TRAIN_IMAGE_DIR,
    train_ann_file=COCO_TRAIN_JSON,
    val_img_dir=COCO_VAL_IMAGE_DIR,
    val_ann_file=COCO_VAL_JSON,
    num_classes=NUM_CLASSES,
    dinov3_repo_path=DINOV3_REPO_PATH,
    dinov3_weights_path=DINOV3_WEIGHTS_PATH,
)

print("patched config saved:", PATCHED_CONFIG_OUT)
print("exists?:", PATCHED_CONFIG_OUT.exists())

with open(PATCHED_CONFIG_OUT, "r", encoding="utf-8") as f:
    print(f.read()[:3000])

patched config saved: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트 미션\초급_프로젝트\RT-DETRv4\configs\rtv4\custom_rtdetrv4_config.yml
exists?: True
__include__:
- ../dfine/dfine_hgnetv2_x_coco.yml
- ../base/rtv4.yml
output_dir: ./outputs/rtv4_hgnetv2_x_coco
teacher_model:
  type: DINOv3TeacherModel
  dinov3_repo_path: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트
    미션\초급_프로젝트\RT-DETRv4\dinov3
  dinov3_weights_path: C:\Users\USER\Desktop\Korea University_Sejong\2025\Codeit_Sprint\스프린트
    미션\초급_프로젝트\RT-DETRv4\dinov3\dinov3\weights\dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth
  patch_size: 16
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
HybridEncoder:
  distill_teacher_dim: 768
RTv4Criterion:
  weight_dict:
    loss_distill: 20
  distill_adaptive_params:
    enabled: true
    rho: 2
    delta: 0.25
    default_weight: 20
optimizer:
  type: AdamW
  params:
  - params: ^(?=.*backbone)(?!.*norm|bn).*$
    lr: 5.0e-06

## 12) Colab에서 바로 학습 실행
Colab Linux에서는 Windows에서 겪었던 분산/libuv 문제를 피하기 쉬워서, 기본 실행은 `torchrun`으로 둔다.


In [37]:
import json
from collections import Counter

ann_path = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/rtdetrv4_work/coco_dataset/annotations/instances_train.json")

print("exists:", ann_path.exists())
print("size(bytes):", ann_path.stat().st_size if ann_path.exists() else "N/A")

with open(ann_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("num_images:", len(coco.get("images", [])))
print("num_annotations:", len(coco.get("annotations", [])))
print("num_categories:", len(coco.get("categories", [])))

if len(coco.get("annotations", [])) > 0:
    print("sample ann:", coco["annotations"][0])
if len(coco.get("categories", [])) > 0:
    print("sample cat:", coco["categories"][0])

exists: True
size(bytes): 5494
num_images: 0
num_annotations: 0
num_categories: 56
sample cat: {'id': 0, 'name': '보령부스파정 5mg'}


In [38]:
import json
from collections import Counter

ann_path = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/rtdetrv4_work/coco_dataset/annotations/instances_train.json"

with open(ann_path, "r", encoding="utf-8") as f:
    coco = json.load(f)

cat_ids = [ann["category_id"] for ann in coco.get("annotations", []) if "category_id" in ann]
unique_cat_ids = sorted(set(cat_ids))

print("num annotations:", len(coco.get("annotations", [])))
print("num unique category ids:", len(unique_cat_ids))

if len(unique_cat_ids) == 0:
    print("category_id가 비어 있음. 즉 annotations가 없거나 category_id가 없는 상태.")
else:
    print("min category_id:", min(unique_cat_ids))
    print("max category_id:", max(unique_cat_ids))
    print("first 20 ids:", unique_cat_ids[:20])

num annotations: 0
num unique category ids: 0
category_id가 비어 있음. 즉 annotations가 없거나 category_id가 없는 상태.


In [18]:
def run_rtdetr_training_colab(
    repo_dir: Path,
    config_path: Path,
    use_amp=True,
    resume_checkpoint=None,
    nproc_per_node=1,
    master_port=7777,
):
    import sys

    train_py = repo_dir / "train.py"
    if not train_py.exists():
        raise FileNotFoundError(f"train.py not found: {train_py}")

    cmd = [
        sys.executable, "-m", "torch.distributed.run",
        "--master_port", str(master_port),
        "--nproc_per_node", str(nproc_per_node),
        str(train_py),
        "-c", str(config_path),
        "--seed", str(SEED),
    ]

    if use_amp:
        cmd.append("--use-amp")

    if resume_checkpoint is not None:
        cmd.extend(["-r", str(resume_checkpoint)])

    env = os.environ.copy()
    env["PYTHONPATH"] = str(repo_dir) + os.pathsep + env.get("PYTHONPATH", "")
    env["MASTER_ADDR"] = "127.0.0.1"
    env["MASTER_PORT"] = str(master_port)
    env["PYTHONIOENCODING"] = "utf-8"

    print("Running command:")
    print(" ".join(cmd))
    print("\nPython:", sys.executable)
    print("Working dir:", repo_dir)

    result = subprocess.run(
        cmd,
        cwd=str(repo_dir),
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )

    print("\n===== STDOUT =====")
    print(result.stdout)
    print("\n===== STDERR =====")
    print(result.stderr)

    return result.returncode

return_code = run_rtdetr_training_colab(
    repo_dir=RTDETR_REPO,
    config_path=PATCHED_CONFIG_OUT,
    use_amp=USE_AMP,
    resume_checkpoint=RESUME_CHECKPOINT,
    nproc_per_node=1,
    master_port=7777,
)

print("return_code:", return_code)

Running command:
/usr/bin/python3 -m torch.distributed.run --master_port 7777 --nproc_per_node 1 /content/RT-DETRv4/train.py -c /content/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml --seed 42 --use-amp

Python: /usr/bin/python3
Working dir: /content/RT-DETRv4

===== STDOUT =====
Initialized distributed mode...
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': None, 'epoches': 58, 'last_epoch': -1, 'lrsheduler': 'flatcosine', 'lr_gamma': 0.5, 'no_aug_epoch': 8, 'warmup_iter': 2000, 'flat_epoch': 29, 'use_amp': True, 'use_ema': Tru

## 13) 체크용 셀


In [ ]:
print("OUT_RAW_ROOT:", OUT_RAW_ROOT)
print("COCO_ROOT:", COCO_ROOT)
print("PATCHED_CONFIG_OUT:", PATCHED_CONFIG_OUT)

print("\n샘플 train 이미지 수:", len(list(COCO_TRAIN_IMAGE_DIR.glob('*'))))
print("샘플 val 이미지 수:", len(list(COCO_VAL_IMAGE_DIR.glob('*'))))

OUT_RAW_ROOT: /content/rtdetrv4_work_colab/augmented_raw
COCO_ROOT: /content/rtdetrv4_work_colab/coco_dataset
PATCHED_CONFIG_OUT: /content/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml

샘플 train 이미지 수: 273
샘플 val 이미지 수: 68


In [ ]:
import torch
from pathlib import Path

ckpt_dir = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/outputs/rtv4_hgnetv2_x_coco")
targets = [
    ckpt_dir / "best_stg1.pth",
    ckpt_dir / "best_stg2.pth",
    ckpt_dir / "last.pth",
]

for p in targets:
    print("=" * 80)
    print("FILE:", p.name)
    if not p.exists():
        print("없음")
        continue

    ckpt = torch.load(p, map_location="cpu")

    print("top-level keys:", list(ckpt.keys())[:20])

    for key in ["epoch", "best_stat", "date", "ema_updates"]:
        if key in ckpt:
            print(f"{key}:", ckpt[key])

    if "stats" in ckpt:
        print("stats:", ckpt["stats"])
    if "args" in ckpt:
        print("args:", ckpt["args"])

FILE: best_stg1.pth
top-level keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_warmup_scheduler']
date: 2026-03-25T05:52:50.071727
FILE: best_stg2.pth
top-level keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_warmup_scheduler']
date: 2026-03-25T06:01:15.910192
FILE: last.pth
top-level keys: ['date', 'last_epoch', 'model', 'criterion', 'postprocessor', 'ema', 'scaler', 'optimizer', 'lr_warmup_scheduler']
date: 2026-03-25T05:52:29.015287


In [ ]:
import subprocess
from pathlib import Path
import re

ckpt_dir = Path("C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/outputs/rtv4_hgnetv2_x_coco")
config_path = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml"

for name in ["best_stg1.pth", "best_stg2.pth", "last.pth"]:
    ckpt_path = ckpt_dir / name
    if not ckpt_path.exists():
        print(f"[SKIP] {name} 없음")
        continue

    print("\n" + "=" * 100)
    print("EVAL START:", ckpt_path.name)
    print("=" * 100)

    cmd = [
        "python", "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/train.py",
        "-c", "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml",
        "--resume", "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/outputs/rtv4_hgnetv2_x_coco/best_stg2.pth",
        "--test-only"
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)

    match = re.search(r"coco_eval_bbox['\"]?\s*[:=]\s*([0-9.]+)", result.stdout)
    if match:
        print("parsed coco_eval_bbox =", float(match.group(1)))
    else:
        print("coco_eval_bbox를 stdout에서 찾지 못함")


EVAL START: best_stg1.pth
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': '/content/RT-DETRv4/outputs/rtv4_hgnetv2_x_coco/best_stg2.pth', 'tuning': None, 'epoches': 58, 'last_epoch': -1, 'lrsheduler': 'flatcosine', 'lr_gamma': 0.5, 'no_aug_epoch': 8, 'warmup_iter': 2000, 'flat_epoch': 29, 'use_amp': False, 'use_ema': True, 'ema_decay': 0.9999, 'ema_warmups': 2000, 'sync_bn': False, 'clip_max_norm': 0.1, 'find_unused_parameters': False, 'seed': None, 'print_freq': 100, 'checkpoint_freq': 4, 'output_dir': './outputs/rtv4_h

## Submission.csv 파일 생성

In [ ]:
import os
import sys
import json
import zipfile
from copy import deepcopy
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image

RTDETR_REPO = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4"
if RTDETR_REPO not in sys.path:
    sys.path.append(RTDETR_REPO)

from engine.core import YAMLConfig

In [ ]:
BEST_PTH_PATH = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/drive/MyDrive/rtdetrv4best.pth"

DATA_ROOT = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/ai09-project01/sprint_ai_project1_data/"
TRAIN_ANNOTATION_DIR = os.path.join(DATA_ROOT, "train_annotations")
TEST_IMAGE_DIR = os.path.join(DATA_ROOT, "test_images")
CONFIG_PATH = "C:/Users/USER/Desktop/Korea University_Sejong/2025/Codeit_Sprint/스프린트 미션/초급_프로젝트/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml"

SUBMISSION_CSV_PATH = "./submission.csv"

CONF_TH = 0.05
IOU_TH = 0.5
MAX_DET = 4
DEVICE = 0  # GPU: 0, CPU: "cpu"

In [ ]:
assert os.path.exists(BEST_PTH_PATH), f"best pth not found: {BEST_PTH_PATH}"
assert os.path.exists(CONFIG_PATH), f"config not found: {CONFIG_PATH}"
assert os.path.isdir(TRAIN_ANNOTATION_DIR), f"train annotation dir not found: {TRAIN_ANNOTATION_DIR}"
assert os.path.isdir(TEST_IMAGE_DIR), f"test image dir not found: {TEST_IMAGE_DIR}"

image_paths = sorted([
    p for p in Path(TEST_IMAGE_DIR).glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
])

print("best.pth:", BEST_PTH_PATH)
print("config:", CONFIG_PATH)
print("train annotation dir:", TRAIN_ANNOTATION_DIR)
print("test image dir:", TEST_IMAGE_DIR)
print("num test images:", len(image_paths))
print("sample test images:", [p.name for p in image_paths[:5]])
print("device:", DEVICE)

best.pth: /content/drive/MyDrive/rtdetrv4best.pth
config: /content/RT-DETRv4/configs/rtv4/custom_rtdetrv4_config.yml
train annotation dir: /content/ai09-project01/sprint_ai_project1_data/train_annotations
test image dir: /content/ai09-project01/sprint_ai_project1_data/test_images
num test images: 842
sample test images: ['1.png', '10.png', '100.png', '1003.png', '1004.png']
device: 0


In [ ]:
def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default


def parse_annotation_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    if isinstance(data, dict) and "annotations" in data:
        categories = data.get("categories", [])
        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in data.get("annotations", []):
            category_id = safe_get(ann, ["category_id"], default=None)
            rows.append({
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")


def build_name_to_original_category_id(annotation_root):
    rows = []
    for root, dirs, files in os.walk(annotation_root):
        for jf in files:
            if not jf.lower().endswith(".json"):
                continue
            json_path = os.path.join(root, jf)
            try:
                rows.extend(parse_annotation_json(json_path))
            except Exception as e:
                print(f"[WARNING] failed to parse {json_path}: {e}")

    ann_df = pd.DataFrame(rows)
    ann_df = ann_df.dropna(subset=["category_name", "category_id"]).drop_duplicates()
    name_to_original_category_id = dict(
        zip(ann_df["category_name"].astype(str), ann_df["category_id"].astype(int))
    )

    print("num original category names:", len(name_to_original_category_id))
    print("sample name -> original category_id:", list(name_to_original_category_id.items())[:10])
    return name_to_original_category_id


def build_rtdetr_model(config_path, checkpoint_path, device):
    cfg = YAMLConfig(config_path, resume=checkpoint_path)

    if "HGNetv2" in cfg.yaml_cfg:
        cfg.yaml_cfg["HGNetv2"]["pretrained"] = False

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    if "ema" in checkpoint:
        state = checkpoint["ema"]["module"]
        print("Using EMA weights from checkpoint.")
    else:
        state = checkpoint["model"]
        print("Using model weights from checkpoint.")

    cfg.model.load_state_dict(state)

    class Model(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.model = cfg.model.deploy()
            self.postprocessor = cfg.postprocessor.deploy()

        def forward(self, images, orig_target_sizes):
            outputs = self.model(images)
            outputs = self.postprocessor(outputs, orig_target_sizes)
            return outputs

    model = Model(cfg).to(device)
    model.eval()
    return model, cfg


def build_model_idx_to_original_category_id(annotation_root, cfg):
    # 원본 annotation json들에서 name -> original category_id
    name_to_original_category_id = build_name_to_original_category_id(annotation_root)

    # RT-DETR 학습용 COCO categories에서 model idx(0~55) -> class name
    train_ann_file = cfg.yaml_cfg["train_dataloader"]["dataset"]["ann_file"]
    with open(train_ann_file, "r", encoding="utf-8") as f:
        coco_train = json.load(f)

    coco_categories = sorted(coco_train["categories"], key=lambda x: x["id"])
    print("num coco categories:", len(coco_categories))
    print("sample coco categories:", coco_categories[:10])

    model_idx_to_original_category_id = {}
    missing_names = []

    for cat in coco_categories:
        model_idx = int(cat["id"])
        class_name = str(cat.get("name", model_idx))

        if class_name in name_to_original_category_id:
            model_idx_to_original_category_id[model_idx] = int(name_to_original_category_id[class_name])
        else:
            missing_names.append((model_idx, class_name))

    print("mapped model classes:", len(model_idx_to_original_category_id))
    print("sample model_idx -> original category_id:", list(model_idx_to_original_category_id.items())[:10])

    if missing_names:
        print("[WARNING] missing class names in annotation mapping:")
        print(missing_names[:20])

    return model_idx_to_original_category_id


model, cfg = build_rtdetr_model(CONFIG_PATH, BEST_PTH_PATH, DEVICE)
model_idx_to_original_category_id = build_model_idx_to_original_category_id(TRAIN_ANNOTATION_DIR, cfg)

print("model loaded.")
print("config num_classes:", cfg.yaml_cfg.get("num_classes"))
print("eval_spatial_size:", cfg.yaml_cfg.get("eval_spatial_size"))

Using EMA weights from checkpoint.
num original category names: 56
sample name -> original category_id: [('글리틴정(콜린알포세레이트)', 33880), ('리바로정 4mg', 29667), ('일양하이트린정 2mg', 3351), ('종근당글리아티린연질캡슐(콜린알포세레이트)\xa0', 18357), ('크레스토정 20mg', 16262), ('로수젯정10/5밀리그램', 36637), ('자누비아정 50mg', 22347), ('트윈스타정 40/5mg', 27733), ('기넥신에프정(은행엽엑스)(수출용)', 3483), ('아모잘탄정 5/100mg', 25469)]
num coco categories: 56
sample coco categories: [{'id': 0, 'name': '보령부스파정 5mg'}, {'id': 1, 'name': '뮤테란캡슐 100mg'}, {'id': 2, 'name': '일양하이트린정 2mg'}, {'id': 3, 'name': '기넥신에프정(은행엽엑스)(수출용)'}, {'id': 4, 'name': '무코스타정(레바미피드)(비매품)'}, {'id': 5, 'name': '알드린정'}, {'id': 6, 'name': '뉴로메드정(옥시라세탐)'}, {'id': 7, 'name': '에어탈정(아세클로페낙)'}, {'id': 8, 'name': '리렉스펜정 300mg/PTP'}, {'id': 9, 'name': '아빌리파이정 10mg'}]
mapped model classes: 56
sample model_idx -> original category_id: [(0, 1900), (1, 2483), (2, 3351), (3, 3483), (4, 3544), (5, 3743), (6, 3832), (7, 4543), (8, 12081), (9, 12247)]
model loaded.
config num_classes: 56
eval_spatial_siz

In [ ]:
transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
])

rows = []
annotation_id = 1
unmapped_pred_classes = set()

for idx, img_path in enumerate(image_paths, start=1):
    image = Image.open(img_path).convert("RGB")
    w, h = image.size

    image_tensor = transform(image).unsqueeze(0).to(DEVICE)
    orig_size = torch.tensor([[w, h]], device=DEVICE)

    with torch.no_grad():
        labels, boxes, scores = model(image_tensor, orig_size)

    labels = labels[0].detach().cpu().numpy().astype(int)
    boxes = boxes[0].detach().cpu().numpy()
    scores = scores[0].detach().cpu().numpy()

    image_id = int(img_path.stem)

    det_count = 0
    for pred_cls, box, score in zip(labels, boxes, scores):
        pred_cls = int(pred_cls)
        score = float(score)

        if score < CONF_TH:
            continue

        if pred_cls not in model_idx_to_original_category_id:
            unmapped_pred_classes.add(pred_cls)
            continue

        x1, y1, x2, y2 = box.tolist()
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(w, x2)
        y2 = min(h, y2)

        bw = max(0, x2 - x1)
        bh = max(0, y2 - y1)

        if bw <= 0 or bh <= 0:
            continue

        original_category_id = int(model_idx_to_original_category_id[pred_cls])

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": original_category_id,
            "bbox_x": int(round(x1)),
            "bbox_y": int(round(y1)),
            "bbox_w": int(round(bw)),
            "bbox_h": int(round(bh)),
            "score": score,
        })
        annotation_id += 1
        det_count += 1

        if det_count >= MAX_DET:
            break

    if idx % 100 == 0 or idx == len(image_paths):
        print(f"[{idx}/{len(image_paths)}] processed")

if unmapped_pred_classes:
    print("[WARNING] unmapped predicted classes:", sorted(unmapped_pred_classes))

[100/842] processed
[200/842] processed
[300/842] processed
[400/842] processed
[500/842] processed
[600/842] processed
[700/842] processed
[800/842] processed
[842/842] processed


In [ ]:
submission_df = pd.DataFrame(rows, columns=[
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
])

submission_df = submission_df.sort_values(
    by=["image_id", "score"],
    ascending=[True, False]
).reset_index(drop=True)

submission_df["annotation_id"] = range(1, len(submission_df) + 1)

submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print("saved:", SUBMISSION_CSV_PATH)
print("num rows:", len(submission_df))
submission_df.head()

saved: ./submission.csv
num rows: 3284


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,596,672,256,481,0.900033
1,2,1,1900,156,251,206,126,0.485310
2,3,1,33208,176,745,175,288,0.271971
3,4,1,29345,176,745,175,288,0.146953
4,5,3,27926,568,628,262,489,0.915157


In [ ]:
print("category_id range:", submission_df["category_id"].min(), submission_df["category_id"].max())
print("bbox_x min:", submission_df["bbox_x"].min(), "bbox_y min:", submission_df["bbox_y"].min())
print("num unique category_ids:", submission_df["category_id"].nunique())
print("sample rows:")
display(submission_df.head())

category_id range: 1900 41768
bbox_x min: 0 bbox_y min: 0
num unique category_ids: 46
sample rows:


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,596,672,256,481,0.900033
1,2,1,1900,156,251,206,126,0.485310
2,3,1,33208,176,745,175,288,0.271971
3,4,1,29345,176,745,175,288,0.146953
4,5,3,27926,568,628,262,489,0.915157
